# Retrieval Tables

This notebook loads saved retrieval and syntax-classification results and builds the LaTeX summary table.

In [ ]:
import os
import sys
from pathlib import Path

RETRIEVAL_DIR = Path.cwd().resolve()
REPO_ROOT = RETRIEVAL_DIR.parent
LOCAL_RESULTS_ROOT = RETRIEVAL_DIR / 'results'
LEGACY_RESULTS_ROOT = REPO_ROOT / 'results'
LOCAL_RESULTS_RECALL_ROOT = LOCAL_RESULTS_ROOT / 'recall'
LEGACY_RESULTS_RECALL_ROOT = LEGACY_RESULTS_ROOT / 'recall'
LOCAL_RESULTS_TABLE = LOCAL_RESULTS_ROOT / 'table'
LEGACY_RESULTS_SYNTAX_CLASSIFICATION = str(LEGACY_RESULTS_ROOT / 'syntax_classification') + '/'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import makefolder
import numpy as np


In [ ]:
model_name = 'gemma12b'
min_token_length = 3
global_center_flag = 1
k_recall = 3
normalization_flag = 1
shuffled_controls = [0, 1]
C = 1000

for shuffled_control in shuffled_controls:
    resultsfolder = makefolder(
        base=LEGACY_RESULTS_SYNTAX_CLASSIFICATION,
        model_name=model_name,
        avg_tokens=0,
        normalization_flag=normalization_flag,
        shuffled_control=shuffled_control,
        C=C,
    )
    results = np.load(resultsfolder + 'results.npz')

    if shuffled_control == 0:
        syn_accs_A = results['accs_A']
        syn_ablated_syn_accs_A = results['syn_ablated_accs_A']
        sem_ablated_syn_accs_A = results['sem_ablated_accs_A']
    elif shuffled_control == 1:
        syn_ablated_syn_accs_A_perm = results['syn_ablated_accs_A']
        sem_ablated_syn_accs_A_perm = results['sem_ablated_accs_A']


In [ ]:
def fmt_num(x):
    return f'{float(x):.2f}'


def style_val(x, max_val, min_val):
    s = fmt_num(x)
    if np.isclose(x, max_val):
        return r'\underline{' + s + '}'
    if np.isclose(x, min_val):
        return r'\textbf{' + s + '}'
    return s


def load_recall_data(var, avg_tokens):
    local_folder = makefolder(
        base=str(LOCAL_RESULTS_RECALL_ROOT / var) + '/',
        create_folder=False,
        model_name=model_name,
        avg_tokens=avg_tokens,
        min_token_length=min_token_length,
        global_center_flag=global_center_flag,
        k=k_recall,
    )
    local_path = os.path.join(local_folder, f'recall_k{k_recall}.npz')
    if os.path.exists(local_path):
        return np.load(local_path)

    legacy_folder = makefolder(
        base=str(LEGACY_RESULTS_RECALL_ROOT / var) + '/',
        create_folder=False,
        model_name=model_name,
        avg_tokens=avg_tokens,
        min_token_length=min_token_length,
        global_center_flag=global_center_flag,
        k=k_recall,
    )
    legacy_path = os.path.join(legacy_folder, f'recall_k{k_recall}.npz')
    return np.load(legacy_path)


best_recalls_by_var = {}
for var in ['syn', 'sem']:
    recall_data = load_recall_data(var, 0 if var == 'syn' else 1)
    best_recalls_by_var[var] = [
        recall_data['recalls_0'].max(),
        recall_data['recalls_sem'].max(),
        recall_data['recalls_syn'].max(),
        recall_data['recalls_sem_perm'].max(),
        recall_data['recalls_syn_perm'].max(),
    ]

best_accs = [
    syn_accs_A.max(),
    sem_ablated_syn_accs_A.max(),
    syn_ablated_syn_accs_A.max(),
    sem_ablated_syn_accs_A_perm.max(),
    syn_ablated_syn_accs_A_perm.max(),
]
best_twin_recalls = best_recalls_by_var['syn']
best_p_recalls = best_recalls_by_var['sem']

row_labels = [
    r'\shortstack{baseline\\(no ablation)}',
    r'\shortstack{semantic\\ablation}',
    r'\shortstack{syntactic\\ablation}',
    r'\shortstack{semantic\\ablation\\(random)}',
    r'\shortstack{syntactic\\ablation\\(random)}',
]

max_acc, min_acc = float(np.max(best_accs)), float(np.min(best_accs))
max_twin, min_twin = float(np.max(best_twin_recalls)), float(np.min(best_twin_recalls))
max_p, min_p = float(np.max(best_p_recalls)), float(np.min(best_p_recalls))

lines = []
lines.append(r'\begin{table}[t]')
lines.append(r'\centering')
lines.append(r'\small')
lines.append(r'\setlength{\tabcolsep}{2pt}')
lines.append(r'\renewcommand{\arraystretch}{1.05}')
lines.append(r'\begin{tabular}{|c|c|c|c|}')
lines.append(r'\hline')
lines.append(
    r' & \shortstack{Best\\syn-acc} & '
    + rf'\shortstack{{Best syn-\\recall@{k_recall}}} & '
    + rf'\shortstack{{Best sem-\\recall@{k_recall}}} \\'
)
lines.append(r'\hline')

for label, a_val, t_val, p_val in zip(row_labels, best_accs, best_twin_recalls, best_p_recalls):
    lines.append(
        f'{label} & '
        f'{style_val(a_val, max_acc, min_acc)} & '
        f'{style_val(t_val, max_twin, min_twin)} & '
        f'{style_val(p_val, max_p, min_p)} \\hline'
    )

lines.append(r'\end{tabular}')
lines.append(r'\label{tab:probes_main}')
lines.append(r'\end{table}')

latex_table = '\n'.join(lines)
print(latex_table)

os.makedirs(LOCAL_RESULTS_TABLE, exist_ok=True)
table_path = LOCAL_RESULTS_TABLE / f'{model_name}_k{k_recall}.txt'
with open(table_path, 'w') as f:
    f.write(latex_table)

print(f'Saved LaTeX table to {table_path}')
